In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### **INSERT LAST_LOAD INTO watermark TABLE (1 times setup)**
---

In [0]:
%sql
INSERT INTO test01.etl.watermark (table_name, last_load) 
SELECT
    'crm_cust_info' AS table_name,
    MIN(_ingested_at) AS last_load
FROM test01.bronze.crm_cust_info;

INSERT INTO test01.etl.watermark (table_name, last_load)
SELECT 
    'crm_prd_info' AS table_name,
    MIN(_ingested_at) AS last_load
FROM test01.bronze.crm_prd_info;

INSERT INTO test01.etl.watermark (table_name, last_load)
SELECT 
    'crm_sales_details' AS table_name,
    MIN(_ingested_at) AS last_load
FROM test01.bronze.crm_sales_details;

INSERT INTO test01.etl.watermark (table_name, last_load)
SELECT
    'erp_cust_az12' AS table_name,
    MIN(_ingested_at) AS last_load
FROM test01.bronze.erp_cust_az12;

INSERT INTO test01.etl.watermark (table_name, last_load)
SELECT
    'erp_loc_a101' AS table_name,
    MIN(_ingested_at) AS last_load
FROM test01.bronze.erp_loc_a101;

INSERT INTO test01.etl.watermark (table_name, last_load)
SELECT
    'erp_px_cat_g1v2' AS table_name,
    MIN(_ingested_at) AS last_load
FROM test01.bronze.erp_px_cat_g1v2

num_affected_rows,num_inserted_rows
1,1


### **Add a SCD TYPE2 col to silver layer**
---

In [0]:
schema = "test01.silver"

columns_to_add = [
    "valid_from TIMESTAMP",
    "valid_to TIMESTAMP",
    "is_used BOOLEAN"
]

tables = spark.sql(f"SHOW TABLES IN {schema}").collect()

for row in tables:
    table_name = row.tableName
    full_name = f"{schema}.{table_name}"
    
    # Build the column list
    col_sql = ", ".join(columns_to_add)
    
    spark.sql(f"""
        ALTER TABLE {full_name}
        ADD COLUMNS ({col_sql})
    """)
    
    print(f"Added columns to {full_name}")

Added columns to test01.silver.crm_cust_info
Added columns to test01.silver.crm_prd_info
Added columns to test01.silver.crm_sales_details
Added columns to test01.silver.erp_cust_az12
Added columns to test01.silver.erp_loc_a101
Added columns to test01.silver.erp_px_cat_g1v2


#### Set a ingested_at date to be a valid_from and set all is_used to be True
---

In [0]:
schema = 'test01.silver'
tables = spark.sql(f'SHOW TABLES IN {schema}').collect()

for row in tables:
    table_name = row.tableName
    full_name = f'{schema}.{table_name}'

    spark.sql(f'''
        UPDATE {full_name}
        SET is_used = true,
            valid_from = CAST(_ingested_at AS DATE)          
    ''')

### **Start cleaning data that came after last_load ( use _ingested_at < last_load )**
---

In [0]:
def get_watermark(table):
    return spark.sql(f'''
                        SELECT last_load FROM test01.etl.watermark
                        WHERE table_name = '{table}'
                     ''').first()[0]
    
def update_watermark(table, view):
    spark.sql(f'''
                UPDATE test01.etl.watermark
                SET last_load = COALESCE((SELECT MAX(_ingested_at) FROM {view}), last_load)
                WHERE table_name = '{table}'
              ''')
    return print(f'Updated watermark for {table} successfully')

#### crm_cust_info
---

In [0]:
def load_crm_cust_info():
    w = Window.partitionBy('cst_id').orderBy(col('cst_create_date').desc())
    last_load = get_watermark('crm_cust_info')

    df = (
            spark.table(f'test01.bronze.crm_cust_info')
            .filter(col('_ingested_at') > last_load)
            .filter(col('cst_id').isNotNull())
            .withColumn('flag_last', row_number().over(w))
            .filter(col('flag_last') == 1)
            .drop(col('flag_last'))
            .withColumn(
                'cst_marital_status', when(trim(upper(col('cst_marital_status'))) == 'M','Marriage')
                    .when(trim(upper(col('cst_marital_status'))) == 'S', 'Single')
                    .otherwise('n/a')            
            )
            .withColumn(
                'cst_gndr', when(trim(upper(col('cst_gndr'))) == 'M', 'Male')
                .when(trim(upper(col('cst_gndr'))) == 'F', 'Female')
                .otherwise('n/a')
            )
            .withColumn('cst_firstname', trim(col('cst_firstname')))
            .withColumn('cst_lastname', trim(col('cst_lastname')))
        )
    
    df.createOrReplaceTempView('source_crm_cust_info')

    spark.sql(f'''
            MERGE INTO test01.silver.crm_cust_info AS t 
            USING source_crm_cust_info AS s
                ON t.cst_id = s.cst_id 
            WHEN MATCHED AND t.is_used = true 
                AND s.cst_create_date > t.cst_create_date -- This will only update the record if new data is newer
                AND NOT ( 
                    s.cst_firstname <=> t.cst_firstname
                    AND s.cst_lastname <=> t.cst_lastname
                    AND s.cst_marital_status <=> t.cst_marital_status
                    AND s.cst_gndr <=> t.cst_gndr
                ) 
            THEN UPDATE SET
                t.is_used = false,
                t.valid_to = s._ingested_at
        ''')
    
    print(f'Add a valid_to and change is_used on crm_cust_info')
    print(f'Start merge new data into crm_cust_info')

    spark.sql(f'''
        INSERT INTO test01.silver.crm_cust_info
        SELECT
            s.cst_id, s.cst_key, s.cst_firstname, s.cst_lastname, s.cst_marital_status, s.cst_gndr, s.cst_create_date, 
            s._ingested_at,
            s._source_file,
            s._ingested_at AS valid_from,
            null AS valid_to,
            true AS is_used
        FROM source_crm_cust_info AS s
        LEFT JOIN test01.silver.crm_cust_info AS t
            ON t.cst_id = s.cst_id
            AND t.is_used = true
        WHERE t.cst_id IS NULL 
    ''')    

    print(f'Inserted data into crm_cust_info successfully')

    update_watermark('crm_cust_info', 'source_crm_cust_info')
    
    print('-' * 60)

#### crm_prd_info
---

In [0]:
def load_crm_prd_info():
    last_load = get_watermark('crm_prd_info')
    w = Window.partitionBy('prd_key').orderBy(col('prd_start_dt').desc())

    df = (
        spark.table('test01.bronze.crm_prd_info')
        .filter(col('_ingested_at') > last_load)
        .filter(col('prd_id').isNotNull())
        .withColumn('cat_id', regexp_replace(substring(col('prd_key'), 1, 5), '-', '_'))
        .withColumn('prd_key', substring(col('prd_key'), 7, length(col('prd_key'))))
        .withColumn('prd_cost', coalesce(col('prd_cost'), lit(0)))
        .withColumn(
            'prd_line', when(upper(trim(col('prd_line'))) == 'R', 'Road')
            .when(upper(trim(col('prd_line'))) == 'T', 'Touring')
            .when(upper(trim(col('prd_line'))) == 'M', 'Mountain')
            .when(upper(trim(col('prd_line'))) == 'S', 'Other Sales')
            .otherwise('n/a')
        )
        .withColumn('rn', row_number().over(w))
    )

    df.createOrReplaceTempView('source_crm_prd_info')

    spark.sql(f'''
        MERGE INTO test01.silver.crm_prd_info AS t
        USING (SELECT * FROM source_crm_prd_info WHERE rn = 1) AS s
            ON t.prd_key = s.prd_key
        WHEN MATCHED AND t.is_used = true 
            AND NOT (
                s.prd_nm <=> t.prd_nm
                AND s.prd_cost <=> t.prd_cost
                AND s.prd_line <=> t.prd_line
                AND s.prd_start_dt <=> t.prd_start_dt
            ) 
        THEN UPDATE SET
            t.is_used = false        
    ''')
    
    print('Add prd_end_date and updated is_used for old data')
    print('Start merge data from source into crm_prd_info')

    spark.sql(f'''
                INSERT INTO test01.silver.crm_prd_info
                (
                    prd_id, prd_key, cat_id, prd_nm, prd_cost, prd_line, prd_start_dt, _ingested_at,
                    _source_file, is_used
                )
                SELECT
                    s.prd_id, s.prd_key, s.cat_id, s.prd_nm, s.prd_cost, s.prd_line, s.prd_start_dt,
                    s._ingested_at, s._source_file, s.rn = 1
                FROM source_crm_prd_info AS s
                LEFT JOIN test01.silver.crm_prd_info AS t
                    ON t.prd_key = s.prd_key
                    AND t.is_used = true
                WHERE t.prd_key IS NULL
              ''')
    
    spark.sql('''
        MERGE INTO test01.silver.crm_prd_info AS t
        USING (
            SELECT 
                prd_id, prd_key, prd_start_dt,
                DATE_ADD(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt), -1) AS new_end_dt
            FROM test01.silver.crm_prd_info
        ) AS s
        ON t.prd_id = s.prd_id AND t.prd_key = s.prd_key AND t.prd_start_dt = s.prd_start_dt
        WHEN MATCHED AND NOT (t.prd_end_dt <=> s.new_end_dt)
        THEN UPDATE SET t.prd_end_dt = s.new_end_dt
    ''')
    
    print('Inserted data into crm_prd_info successfully')
    update_watermark('crm_prd_info', 'source_crm_prd_info')
    print('-' * 60)

#### sales_details
---

In [0]:
def load_crm_sales_details():
    last_load = get_watermark('crm_sales_details')

    df = (
        spark.table('test01.bronze.crm_sales_details')
        .filter(col('_ingested_at') > last_load)
        .withColumn(
            'sls_order_dt', when(length(col('sls_order_dt')) != 8, lit(None))
            .otherwise(to_date(col('sls_order_dt'), 'yyyyMMdd'))
        )
        .withColumn(
            'sls_ship_dt', when(length(col('sls_ship_dt')) != 8, lit(None))
            .otherwise(to_date(col('sls_ship_dt'), 'yyyyMMdd'))
        )
        .withColumn(
            'sls_due_dt', when(length(col('sls_due_dt')) != 8, lit(None))
            .otherwise(to_date(col('sls_due_dt'), 'yyyyMMdd'))
        )
        .withColumn(
            'sls_sales', when((col('sls_sales') <= 0) | (col('sls_sales').isNull()) | (col('sls_sales') != col('sls_quantity') * abs(col('sls_price'))), col('sls_quantity') * abs(col('sls_price')))
            .otherwise(col('sls_sales'))
        )
        .withColumn(
            'sls_price', when((col('sls_price') <= 0) | (col('sls_price').isNull()), col('sls_sales') / col('sls_quantity'))
            .otherwise(col('sls_price'))
        )
    )

    df.createOrReplaceTempView('source_crm_sales_details')

    spark.sql('''
        MERGE INTO test01.silver.crm_sales_details AS t
        USING source_crm_sales_details AS s
            ON t.sls_ord_num = s.sls_ord_num
        WHEN MATCHED AND t.is_used = true
            AND NOT (
                s.sls_prd_key <=> t.sls_prd_key
                AND s.sls_cst_id <=> t.sls_cst_id
                AND s.sls_order_dt <=> t.sls_order_dt  
                AND s.sls_ship_dt <=> t.sls_ship_dt
                AND s.sls_due_dt <=> t.sls_due_dt
                AND s.sls_sales <=> t.sls_sales
                AND s.sls_quantity <=> t.sls_quantity
                AND s.sls_price <=> t.sls_price
            )
        THEN UPDATE SET
            t.is_used = false,
            t.valid_to = s._ingested_at
    ''')

    print('Add valid_from and updated is_used for old data')
    print('Start merge data from source into crm_sales_details')

    spark.sql('''
        INSERT INTO test01.silver.crm_sales_details(
            sls_ord_num, sls_prd_key, sls_cst_id, sls_order_dt, sls_ship_dt, sls_due_dt, sls_sales, sls_quantity, sls_price, _ingested_at,
            _source_file, valid_from, valid_to, is_used
        )
        SELECT
            s.sls_ord_num, s.sls_prd_key, s.sls_cst_id, s.sls_order_dt, s.sls_ship_dt, s.sls_due_dt, s.sls_sales, s.sls_quantity, s.sls_price, s._ingested_at, s._source_file, s._ingested_at, NULL, TRUE
        FROM source_crm_sales_details AS s
        LEFT JOIN test01.silver.crm_sales_details AS t
            ON t.sls_ord_num = s.sls_ord_num
            AND t.is_used = true
        WHERE t.sls_ord_num IS NULL       
    ''')

    print('Inserted data into crm_sales_details successfully')

    update_watermark('crm_sales_details', 'source_crm_sales_details')

    print('-' * 60)

#### erp_cust_az12
---

In [0]:
def load_erp_cust_az12():
    last_load = get_watermark('erp_cust_az12')
    w = Window.partitionBy('cid').orderBy(col('_ingested_at').desc())

    df = (
        spark.table('test01.bronze.erp_cust_az12')
        .filter(col('_ingested_at') > last_load)
        .withColumn(
            'cid', when(col('cid').startswith('NAS'), substring(col('cid'), 4, length(col('cid'))))
            .otherwise(col('cid'))        
        )
        .withColumn(
            'bdate', when(col('bdate') > current_date(), lit(None))
            .otherwise(col('bdate'))
        )
        .withColumn(
            'gen', when(trim(col('gen')).isin('M', 'Male'), 'Male')
            .when(trim(col('gen')).isin('F', 'Female'), 'Female')
            .otherwise('n/a')
        )
        .withColumn(
            'rn', row_number().over(w)
        )
    )

    df.createOrReplaceTempView('source_erp_cust_az12')

    spark.sql('''
        MERGE INTO test01.silver.erp_cust_az12 AS t
        USING (SELECT * FROM source_erp_cust_az12 WHERE rn = 1) AS s
            ON t.cid = s.cid
        WHEN MATCHED AND t.is_used = true
            AND NOT (
                s.bdate <=> t.bdate
                AND s.gen <=> t.gen
            )
        THEN UPDATE SET
            t.is_used = false        
    ''')

    print('Add valid_from and updated is_used for old data')
    print('Start merge data from source into erp_cust_az12')

    spark.sql('''
        INSERT INTO test01.silver.erp_cust_az12(
            cid, bdate, gen, _ingested_at, _source_file, valid_from, valid_to, is_used
        )
        SELECT
            s.cid, s.bdate, s.gen, s._ingested_at, s._source_file, s._ingested_at, NULL, s.rn=1
        FROM source_erp_cust_az12 AS s
        LEFT JOIN test01.silver.erp_cust_az12 AS t
            ON t.cid = s.cid
            AND t.is_used = true
        WHERE t.cid IS NULL       
    ''')

    spark.sql('''
        MERGE INTO test01.silver.erp_cust_az12 AS t
        USING (
            SELECT 
                cid, valid_from,
                LEAD(valid_from) OVER(PARTITION BY cid ORDER BY valid_from) AS next_from
            FROM test01.silver.erp_cust_az12
        ) AS s
            ON t.cid = s.cid AND t.valid_from = s.valid_from
        WHEN MATCHED AND s.next_from IS NOT NULL
            AND NOT (t.valid_to <=> s.next_from - INTERVAL 1 DAY)
        THEN UPDATE SET
            t.valid_to = s.next_from - INTERVAL 1 DAY
    ''')

    print('Inserted data into erp_cust_az12 successfully')

    update_watermark('erp_cust_az12', 'source_erp_cust_az12')

    print('-' * 60)

#### erp_loc_a101
---

In [0]:
def load_erp_loc_a101():
    last_load = get_watermark('erp_loc_a101')
    w = Window.partitionBy('cid').orderBy(col('_ingested_at').desc())

    df = (
        spark.table('test01.bronze.erp_loc_a101')
        .filter(col('_ingested_at') > last_load)
        .withColumn('cid', regexp_replace(col('cid'), '-', ''))
        .withColumn(
            'cntry', when(trim(col('cntry')).isin('US', 'USA', 'United States'), 'United States')
            .when(trim(col('cntry')).isin('DE', 'Germany'), 'Germany')
            .when((trim(col('cntry')).isNull()) | (trim(col('cntry')) == ''), 'n/a')
            .otherwise(col('cntry'))
        )
        .withColumn('rn', row_number().over(w))
    )

    df.createOrReplaceTempView('source_erp_loc_a101')

    spark.sql('''
        MERGE INTO test01.silver.erp_loc_a101 AS t
        USING (SELECT * FROM source_erp_loc_a101 WHERE rn = 1) AS s
            ON t.cid = s.cid
        WHEN MATCHED AND t.is_used = TRUE
            AND NOT (t.cntry <=> s.cntry)
        THEN UPDATE SET
            t.is_used = FALSE     
    ''')

    print('Add valid_from and updated is_used for old data')
    print('Start merge data from source into erp_loc_a101')

    spark.sql('''
        INSERT INTO test01.silver.erp_loc_a101 (
            cid, cntry, _ingested_at, _source_file, valid_from, valid_to, is_used
        )          
        SELECT 
            s.cid, s.cntry, s._ingested_at, s._source_file, s._ingested_at, NULL, TRUE
        FROM source_erp_loc_a101 AS s
        LEFT JOIN test01.silver.erp_loc_a101 AS t
            ON s.cid = t.cid
            AND t.is_used = TRUE
        WHERE t.cid IS NULL 
    ''')

    spark.sql('''
        MERGE INTO test01.silver.erp_loc_a101 AS t
        USING (
            SELECT cid, valid_from,
                LEAD(valid_from) OVER (PARTITION BY cid ORDER BY valid_from) AS next_from
            FROM test01.silver.erp_loc_a101
        ) AS s
            ON t.cid = s.cid AND t.valid_from = s.valid_from
        WHEN MATCHED AND s.next_from IS NOT NULL
            AND NOT (t.valid_to <=> s.next_from - INTERVAL 1 DAY)
        THEN UPDATE SET t.valid_to = s.next_from - INTERVAL 1 DAY
    ''')

    print('Inserted data into erp_loc_a101 successfully')

    update_watermark('erp_loc_a101', 'source_erp_loc_a101')

    print('-' * 60)

#### erp_px_cat_g1v2
---

In [0]:
def load_erp_px_cat_g1v2():
    last_load = get_watermark('erp_px_cat_g1v2')
    w = Window.partitionBy('id').orderBy(col('_ingested_at').desc())

    df = (
        spark.table('test01.bronze.erp_px_cat_g1v2')
        .filter(col('_ingested_at') > last_load)
        .withColumn('rn', row_number().over(w))
    )

    df.createOrReplaceTempView('source_erp_px_cat_g1v2')

    spark.sql('''
        MERGE INTO test01.silver.erp_px_cat_g1v2 AS t
        USING (SELECT * FROM source_erp_px_cat_g1v2 WHERE rn = 1) AS s
            ON s.id = t.id
        WHEN MATCHED AND t.is_used = TRUE
            AND NOT (
                s.cat <=> t.cat
                AND s.subcat <=> t.subcat
                AND s.maintenance <=> t.maintenance
            )
        THEN UPDATE SET
            t.is_used = FALSE     
    ''')

    print('Add valid_from and updated is_used for old data')
    print('Start merge data from source into erp_px_cat_g1v2')

    spark.sql('''
        INSERT INTO test01.silver.erp_px_cat_g1v2 (
            id, cat, subcat, maintenance, _ingested_at, _source_file, valid_from, is_used
        )
        SELECT
            s.id, s.cat, s.subcat, s.maintenance, s._ingested_at, s._source_file, s._ingested_at, s.rn = 1
        FROM source_erp_px_cat_g1v2 AS s
        LEFT JOIN test01.silver.erp_px_cat_g1v2 AS t
            ON s.id = t.id
            AND t.is_used = TRUE
        WHERE t.id IS NULL
    ''')

    spark.sql('''
        MERGE INTO test01.silver.erp_px_cat_g1v2 AS t
        USING (
            SELECT
                id, valid_from,
                LEAD(valid_from) OVER(PARTITION BY id ORDER BY valid_from) AS next_from
            FROM test01.silver.erp_px_cat_g1v2
        ) AS s
        ON t.id = s.id AND t.valid_from = s.valid_from
        WHEN MATCHED AND s.next_from IS NOT NULL
            AND NOT (t.valid_to <=> s.next_from - INTERVAL 1 DAY)
        THEN UPDATE SET
            t.valid_to = s.next_from - INTERVAL 1 DAY
    ''')

    print('Inserted data into erp_px_cat_g1v2 successfully')

    update_watermark('erp_px_cat_g1v2', 'source_erp_px_cat_g1v2')

    print('-' * 60)

### **Start Transforms and Load Data into Silver Layer**
---

In [0]:
load_crm_cust_info()
print('\n')
load_crm_prd_info()
print('\n')
load_crm_sales_details()
print('\n')
load_erp_cust_az12()
print('\n')
load_erp_loc_a101()
print('\n')
load_erp_px_cat_g1v2()